# Validación end-to-end de GeoCrop Analysis MX

Este cuaderno valida **todo el flujo del proyecto** paso a paso, desde la verificación del
entorno hasta el mapa de clasificación final, usando el caso de prueba incluido en el
repositorio (Valle del Yaqui, Sonora).

**Requisitos previos** (ver el tutorial en PDF para el detalle):
1. Repositorio clonado y entorno virtual creado con `pip install -r requirements.txt`.
2. Carpetas `data/` y `outputs/` como hermanas del repositorio.
3. (Opcional pero recomendado) token de NASA Earthdata en el archivo `env`.

> Ejecuta este cuaderno **desde la raíz del repositorio** (`geocrop_analysis_mx/`).


In [ ]:
# Paso 0 - ¿Dónde estamos y con qué Python?
import sys, platform, os
print(f"Python    : {sys.version.split()[0]}")
print(f"Plataforma: {platform.system()} {platform.machine()}")
print(f"Directorio: {os.getcwd()}")
assert os.path.exists("src/main.py"), "Ejecuta este cuaderno desde la raíz del repositorio"


In [ ]:
# Paso 1 - Validar el entorno (mismas verificaciones que `python check_env.py`)
!{sys.executable} check_env.py


## Fase de datos de prueba

`setup_test` copia el AOI, las etiquetas de campo y los mosaicos satelitales pre-procesados
a las carpetas `data/` y `outputs/`. Para que esta validación sea reproducible, primero
eliminamos los productos derivados de corridas anteriores (conservando los mosaicos).


In [ ]:
import shutil, glob
for sub in ["segmentation/segmented_clumps_test.tif", "segmentation/segmented_polygons_test.*",
            "labeling", "features_test.csv", "modeling"]:
    for path in glob.glob(f"../outputs/aoi_yaqui_test/{sub}"):
        (shutil.rmtree if os.path.isdir(path) else os.remove)(path)
print("Productos derivados eliminados; mosaicos conservados.")
!{sys.executable} src/main.py --config config.test.yaml --phase setup_test


## Demostración de descarga real (opcional, requiere internet)

Antes de las fases offline, demostramos el **reemplazo de Google Earth Engine**: buscamos
escenas HLS en el catálogo STAC de la NASA, transmitimos únicamente la ventana del AOI de
cada COG y calculamos la **geomediana** localmente. Observa los mensajes de progreso con
estimación de tiempo restante — el mismo comportamiento que verás en descargas grandes.

Usamos un AOI pequeño (~4×4 km) y un solo mes para que tarde ~2 minutos.


In [ ]:
sys.path.insert(0, "src")
from config import load_config; load_config("config.test.yaml")   # carga el archivo `env` si existe
from data_download import stac_utils, stac_multispectral
from odc.geo.geobox import GeoBox

stac_utils.configure_gdal_env()
proveedor, catalogo = stac_multispectral.resolve_provider("auto")
print(f"Proveedor óptico: {proveedor}")

bbox = (-109.76, 27.28, -109.72, 27.32)
geobox = GeoBox.from_bbox(bbox, crs="EPSG:4326", resolution=30 / stac_utils.METERS_PER_DEGREE)
aoi_demo = {"type": "Polygon", "coordinates": [[
    [bbox[0], bbox[1]], [bbox[2], bbox[1]], [bbox[2], bbox[3]], [bbox[0], bbox[3]], [bbox[0], bbox[1]]]]}

items = stac_multispectral.search_hls_items(catalogo, "2018-03-01", "2018-03-31", aoi_demo, proveedor)
print("Escenas encontradas:", {k: len(v) for k, v in items.items()})

if proveedor == "nasa":
    with stac_utils.earthdata_gdal_session(stac_utils.earthdata_token()):
        stack = stac_multispectral.load_hls_stack(items, geobox, proveedor)
else:
    stack = stac_multispectral.load_hls_stack(items, geobox, proveedor)
compuesto = stac_multispectral.build_composite(stack)
print("Geomediana lista:", compuesto.shape)


In [ ]:
# Visualizar la geomediana descargada
import numpy as np
import matplotlib.pyplot as plt

rgb = np.clip(np.dstack([compuesto[2], compuesto[1], compuesto[0]]) / 3000.0, 0, 1)
ndvi = compuesto[6] / 10000.0
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
ax1.imshow(rgb); ax1.set_title("RGB - geomediana marzo 2018"); ax1.axis("off")
im = ax2.imshow(ndvi, cmap="RdYlGn", vmin=0, vmax=0.9)
ax2.set_title("NDVI"); ax2.axis("off"); plt.colorbar(im, ax=ax2, shrink=0.8)
plt.tight_layout(); plt.show()


## Fases del pipeline (offline, con los mosaicos de prueba)

Cada fase es un comando de consola. Aquí las ejecutamos una por una para inspeccionar sus
salidas; `full_run` las encadenaría todas.


In [ ]:
# Fase 2 - Segmentación Shepherd (shepherd-wasm: NumPy/SciPy puro)
!{sys.executable} src/main.py --config config.test.yaml --phase segment


In [ ]:
# Fase 3 - Etiquetado (filtro de pureza de segmentos)
!{sys.executable} src/main.py --config config.test.yaml --phase label


In [ ]:
# Fase 4 - Extracción de características (exactextract, ~1 min)
!{sys.executable} src/main.py --config config.test.yaml --phase extract


In [ ]:
# Fase 5 - Entrenamiento AutoML con TPOT (~1 min con la config de prueba)
!{sys.executable} src/main.py --config config.test.yaml --phase train


In [ ]:
# Fase 6 - Predicción y mapa final
!{sys.executable} src/main.py --config config.test.yaml --phase predict


## Resultado final

El producto principal es un GeoPackage con la clase de cultivo predicha para **cada
segmento** del AOI.


In [ ]:
# Reporte de clasificación del modelo
print(open("../outputs/aoi_yaqui_test/modeling/classification_report.txt").read())


In [ ]:
# Mapa final de clasificación
import geopandas as gpd
import matplotlib.pyplot as plt

mapa = gpd.read_file("../outputs/aoi_yaqui_test/modeling/predicted_map_test.gpkg")
col = "predicted_label" if "predicted_label" in mapa.columns else mapa.columns[-2]
print(f"{len(mapa)} segmentos clasificados. Columnas: {list(mapa.columns)}")
fig, ax = plt.subplots(figsize=(12, 7))
mapa.plot(column=col, legend=True, ax=ax, linewidth=0,
          legend_kwds={"loc": "lower right", "fontsize": 8})
ax.set_title("Mapa de clasificación de cultivos - Valle del Yaqui (caso de prueba)")
ax.set_axis_off()
plt.tight_layout(); plt.show()


## Validación completada

Si llegaste hasta aquí sin errores, tu instalación es plenamente funcional:

| Componente | Validado con |
|---|---|
| Entorno pip | `check_env.py` |
| Descarga STAC/COG + geomediana | demo online (fase download) |
| Segmentación (shepherd-wasm) | fase `segment` |
| Etiquetado y features (exactextract) | fases `label` y `extract` |
| AutoML (TPOT) | fase `train` |
| Mapa final | fase `predict` |

Para clasificar **tu propia región**, sigue la guía `NEW_AOI.es.md` del repositorio.
